# TV4 – Basic Models (sau khi cập nhật TV3)

Notebook này thực hiện phần mô hình cơ bản cho bài toán **Predicting Product Sales**.

**Dữ liệu sử dụng**
- `data/processed/tv3_train.csv`
- `data/processed/tv3_validation.csv`

**Các mô hình**
- Baseline Mean và Baseline Median
- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression

**Đánh giá:** MAE, RMSE, R² và TimeSeriesSplit theo tuần.

> Lưu ý: Notebook chỉ dùng Train để huấn luyện/CV và Validation để so sánh mô hình. Không đọc hoặc đánh giá `tv3_test.csv`; phần Test để TV6 thực hiện ở bước cuối.
>
> Các đặc trưng RFM được tính từ dữ liệu của chính tuần dự đoán bị loại khỏi mô hình để hạn chế rò rỉ dữ liệu tương lai.

In [ ]:
# 1. Import thư viện
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


In [ ]:
# 2. Đọc dữ liệu TV3 đã xử lý
# Notebook nằm trong thư mục notebooks/, dữ liệu nằm trong data/processed/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH = DATA_DIR / "tv3_train.csv"
VALID_PATH = DATA_DIR / "tv3_validation.csv"

if not TRAIN_PATH.exists() or not VALID_PATH.exists():
    raise FileNotFoundError(
        "Không tìm thấy tv3_train.csv hoặc tv3_validation.csv. "
        "Hãy kiểm tra chúng nằm trong data/processed/."
    )

train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
display(train_df.head())


In [ ]:
# 3. Kiểm tra dữ liệu và xác định cột mục tiêu
TARGET = "sales"
TIME_COL = "week_start"

if TARGET not in train_df.columns or TARGET not in valid_df.columns:
    raise ValueError(f"Không tìm thấy cột mục tiêu '{TARGET}'. Hãy kiểm tra file TV3.")

if TIME_COL not in train_df.columns or TIME_COL not in valid_df.columns:
    raise ValueError(f"Không tìm thấy cột thời gian '{TIME_COL}'. Hãy kiểm tra file TV3.")

train_df[TIME_COL] = pd.to_datetime(train_df[TIME_COL], errors="coerce")
valid_df[TIME_COL] = pd.to_datetime(valid_df[TIME_COL], errors="coerce")

train_df = train_df.dropna(subset=[TARGET, TIME_COL]).copy()
valid_df = valid_df.dropna(subset=[TARGET, TIME_COL]).copy()

print("Khoảng thời gian Train:", train_df[TIME_COL].min(), "->", train_df[TIME_COL].max())
print("Khoảng thời gian Validation:", valid_df[TIME_COL].min(), "->", valid_df[TIME_COL].max())
print("Số dòng Train:", len(train_df))
print("Số dòng Validation:", len(valid_df))


In [ ]:
# 4. Tách X/y và loại các cột không được dùng làm đặc trưng
# Không đưa target, thời gian dự đoán và các biến RFM của cùng tuần vào mô hình.
rfm_exclude = {
    "avg_customer_prior_recency_days",
    "avg_customer_prior_frequency",
    "avg_customer_prior_monetary",
    "R_score", "F_score", "M_score", "RFM_score",
}

# Loại thêm các biến định danh dòng nếu có; mã sản phẩm (ví dụ stockcode)
# vẫn được giữ lại để mô hình có thể học khác biệt giữa các sản phẩm.
drop_cols = {TARGET, TIME_COL} | rfm_exclude

feature_cols = [c for c in train_df.columns if c not in drop_cols]

# Chỉ giữ các cột có mặt ở cả Train và Validation
feature_cols = [c for c in feature_cols if c in valid_df.columns]

X_train = train_df[feature_cols].copy()
y_train = pd.to_numeric(train_df[TARGET], errors="coerce")
X_valid = valid_df[feature_cols].copy()
y_valid = pd.to_numeric(valid_df[TARGET], errors="coerce")

# Đồng bộ kiểu dữ liệu theo Train; chuyển cột ngày còn lại thành dạng số nếu có
for col in feature_cols:
    if pd.api.types.is_datetime64_any_dtype(X_train[col]):
        X_train[col] = X_train[col].astype("int64") / 10**9
        X_valid[col] = pd.to_datetime(X_valid[col], errors="coerce").astype("int64") / 10**9

valid_target_mask = y_valid.notna()
X_valid = X_valid.loc[valid_target_mask]
y_valid = y_valid.loc[valid_target_mask]

print("Số đặc trưng:", len(feature_cols))
print("Các đặc trưng:", feature_cols)
print("Đã loại các cột RFM hiện tại nếu có:", sorted(rfm_exclude.intersection(train_df.columns)))


In [ ]:
# 5. Xây dựng bước tiền xử lý
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_features),
        ("cat", categorical_preprocessor, categorical_features)
    ],
    remainder="drop"
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 6. Đánh giá bằng TimeSeriesSplit trên Train

Các dòng được chia theo **tuần**, không chia ngẫu nhiên theo từng dòng sản phẩm. Nhờ đó, các tuần validation của mỗi fold nằm sau các tuần train của fold đó.

In [ ]:
# 6. Tạo các fold theo thứ tự thời gian
# Mỗi tuần được xem là một nhóm để không tách các sản phẩm cùng tuần sang hai phía.
train_for_cv = train_df.loc[y_train.notna()].copy()
train_for_cv["_target_numeric"] = pd.to_numeric(train_for_cv[TARGET], errors="coerce")
train_for_cv = train_for_cv.dropna(subset=["_target_numeric", TIME_COL])

weeks = np.array(sorted(train_for_cv[TIME_COL].dropna().unique()))
n_splits = min(5, len(weeks) - 1)

if n_splits < 2:
    raise ValueError("Không đủ số tuần trong Train để thực hiện TimeSeriesSplit.")

tscv = TimeSeriesSplit(n_splits=n_splits)
cv_splits = []

for train_week_idx, val_week_idx in tscv.split(weeks):
    fold_train_weeks = set(weeks[train_week_idx])
    fold_val_weeks = set(weeks[val_week_idx])

    fold_train_idx = np.flatnonzero(train_for_cv[TIME_COL].isin(fold_train_weeks).to_numpy())
    fold_val_idx = np.flatnonzero(train_for_cv[TIME_COL].isin(fold_val_weeks).to_numpy())

    cv_splits.append((fold_train_idx, fold_val_idx))

print(f"Số fold: {len(cv_splits)}")
for i, (tr_idx, va_idx) in enumerate(cv_splits, start=1):
    print(
        f"Fold {i}: train đến {train_for_cv.iloc[tr_idx][TIME_COL].max().date()} | "
        f"validation {train_for_cv.iloc[va_idx][TIME_COL].min().date()} -> "
        f"{train_for_cv.iloc[va_idx][TIME_COL].max().date()}"
    )


In [ ]:
# 7. Khai báo mô hình
models = {
    "Baseline Mean": DummyRegressor(strategy="mean"),
    "Baseline Median": DummyRegressor(strategy="median"),
    "Linear Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LinearRegression())
    ]),
    "Ridge": Pipeline([
        ("preprocess", preprocessor),
        ("model", Ridge(alpha=1.0))
    ]),
    "Lasso": Pipeline([
        ("preprocess", preprocessor),
        ("model", Lasso(alpha=0.001, max_iter=10000))
    ]),
    "Decision Tree": Pipeline([
        ("preprocess", preprocessor),
        ("model", DecisionTreeRegressor(
            max_depth=10,
            min_samples_split=10,
            min_samples_leaf=5,
            random_state=RANDOM_STATE
        ))
    ])
}


In [ ]:
# 8. Cross-validation trên Train
def calculate_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

cv_rows = []

X_cv = train_for_cv[feature_cols].copy()
y_cv = train_for_cv["_target_numeric"].copy()

for model_name, model in models.items():
    fold_metrics = []
    for fold_no, (tr_idx, va_idx) in enumerate(cv_splits, start=1):
        X_tr, X_va = X_cv.iloc[tr_idx], X_cv.iloc[va_idx]
        y_tr, y_va = y_cv.iloc[tr_idx], y_cv.iloc[va_idx]

        model.fit(X_tr, y_tr)
        pred = model.predict(X_va)
        fold_metrics.append(calculate_metrics(y_va, pred))

    cv_rows.append({
        "model": model_name,
        "CV_MAE": np.mean([m["MAE"] for m in fold_metrics]),
        "CV_RMSE": np.mean([m["RMSE"] for m in fold_metrics]),
        "CV_R2": np.mean([m["R2"] for m in fold_metrics])
    })

cv_results = pd.DataFrame(cv_rows).sort_values("CV_RMSE")
display(cv_results)


## 9. Huấn luyện trên toàn bộ Train và đánh giá Validation

Sau khi so sánh cross-validation, mỗi mô hình được huấn luyện lại trên toàn bộ Train và đánh giá trên Validation. Validation dùng để lựa chọn mô hình; không dùng Test ở bước này.

In [ ]:
# 9. Validation evaluation
validation_rows = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    valid_pred = model.predict(X_valid)
    metrics = calculate_metrics(y_valid, valid_pred)

    validation_rows.append({
        "model": model_name,
        "Validation_MAE": metrics["MAE"],
        "Validation_RMSE": metrics["RMSE"],
        "Validation_R2": metrics["R2"]
    })

validation_results = pd.DataFrame(validation_rows).sort_values("Validation_RMSE")
display(validation_results)


In [ ]:
# 10. Tổng hợp kết quả và lưu CSV
results_basic = cv_results.merge(validation_results, on="model", how="inner")
results_basic = results_basic.sort_values("Validation_RMSE").reset_index(drop=True)

display(results_basic)

RESULT_PATH = PROJECT_ROOT / "results_basic.csv"
results_basic.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")
print("Đã lưu kết quả tại:", RESULT_PATH)


## 11. Nhận xét kết quả

Sau khi chạy notebook, dựa vào bảng `results_basic` để viết nhận xét:

- So sánh Baseline Mean/Median với các mô hình học máy để xem mô hình có cải thiện so với dự đoán cơ sở hay không.
- MAE và RMSE càng thấp thì sai số dự đoán càng nhỏ.
- R² càng cao thì mô hình giải thích biến thiên của doanh số càng tốt; R² âm cho thấy mô hình kém hơn dự đoán bằng giá trị trung bình trên tập đánh giá.
- Ưu tiên xem kết quả Validation để lựa chọn mô hình, đồng thời đối chiếu với kết quả CV theo thời gian.
- Không kết luận mô hình tốt nhất trên Test trong phần TV4. Tập Test được giữ lại cho bước đánh giá cuối cùng của nhóm.

**Lưu ý khi báo cáo:** Không tự điền số liệu trước khi chạy notebook. Dùng đúng các giá trị được tạo trong `results_basic.csv`.